# 02 - Preprocessing Outlier

Notebook này thực hiện các bước preprocessing dữ liệu cho dự án phân tích tuổi abalone (AbaloneAge). Tập trung vào xử lý biến categorical, outlier detection và transformation để chuẩn bị dữ liệu cho modeling.

### Các bước chính
1. **Import và Setup**: Import thư viện cần thiết và thiết lập đường dẫn dự án.
2. **Load Dữ liệu**: Tải dữ liệu từ file CSV đã clean.
3. **One-Hot Encoding**: Mã hóa biến `Sex` thành các cột binary (Sex_F, Sex_I, Sex_M).
4. **Outlier Clipping**: Sử dụng phương pháp IQR để clip outliers trên các biến numeric (size và weight).
5. **Robust Log Transform**: Áp dụng log transformation để ổn định variance và giảm skewness.
6. **Lưu Dữ liệu**: Xuất hai tập dữ liệu processed vào thư mục `data/processed`.

### Chuẩn bị thư viện

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

from src.data.load_data import ABALONE_COLUMNS, load_abalone_data
from src.data.clean_data import clean_abalone_data, summarize_data_quality
from src.features.feature_engineering import (
    apply_log1p_to_columns,
    build_encoded_preprocessor,
    build_robust_scaled_preprocessor,
    build_standard_scaled_preprocessor,
    prepare_abalone_feature_groups,
    transform_with_preprocessor,
)

# Cấu hình hiển thị
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path.cwd().resolve()
if 'notebooks' in str(PROJECT_ROOT):
    while PROJECT_ROOT.name != 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

### 1. Nạp dữ liệu

In [30]:
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'

# Goi ham load_abalone_data() trong src/data/load_data.py
columns = ABALONE_COLUMNS.copy()
df = load_abalone_data(DATA_PATH, columns=columns)
display(df.head())

,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


### 2. Các nhóm đặc trưng

In [31]:
# Dùng prepare_abalone_feature_groups() từ src/features/feature_engineering.py
feature_groups = prepare_abalone_feature_groups()
target_col = feature_groups['target_col']
categorical_cols = feature_groups['categorical_cols']
size_cols = feature_groups['size_cols']
weight_cols = feature_groups['weight_cols']
numeric_cols = feature_groups['numeric_cols']

print("Biến phân loại:", categorical_cols)
print("Biến kích thước:", size_cols)
print("Biến khối lượng:", weight_cols)
print("Biến mục tiêu:", target_col)


Biến phân loại: ['Sex']
Biến kích thước: ['Length', 'Diameter', 'Height']
Biến khối lượng: ['WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight']
Biến mục tiêu: Rings


### 3. Làm sạch dữ liệu

In [32]:
# Dùng summarize_data_quality() và clean_abalone_data() từ src/data/clean_data.py
quality_summary = summarize_data_quality(df)
df_clean = clean_abalone_data(
    df,
    normalize_columns=False,
    drop_duplicates=True,
    strip_categorical_values=True, 
)
duplicate_count_after = int(df_clean.duplicated().sum())

print(f"Tổng số giá trị thiếu: {quality_summary['missing_count']}")
print(f"Số dòng trùng lặp trước khi làm sạch: {quality_summary['duplicate_count']}")
print(f"Số dòng trùng lặp sau khi làm sạch: {duplicate_count_after}")
print(f"Kích thước dữ liệu sau làm sạch: {df_clean.shape}")


Tổng số giá trị thiếu: 0
Số dòng trùng lặp trước khi làm sạch: 0
Số dòng trùng lặp sau khi làm sạch: 0
Kích thước dữ liệu sau làm sạch: (4177, 9)


### 3. One-hot encoder

In [33]:
def one_hot_encode_sex(df: pd.DataFrame, drop_first: bool = False) -> pd.DataFrame:
    """One-hot encode biến Sex và trả về DataFrame với các cột mới."""
    sex_index = df.columns.get_loc('Sex')
    encoded_df = pd.get_dummies(df['Sex'], prefix='Sex', drop_first=drop_first).astype(int)
    
    # Chèn các cột mới vào vị trí cũ của 'Sex'
    result_df = df.drop('Sex', axis=1)
    for i, col in enumerate(encoded_df.columns):
        result_df.insert(sex_index + i, col, encoded_df[col])
    
    return result_df


encoded = one_hot_encode_sex(df, drop_first=False)
print('Columns:', encoded.columns.tolist())
print('Shape:', encoded.shape)
display(encoded.head())

Columns: ['Sex_F', 'Sex_I', 'Sex_M', 'Length', 'Diameter', 'Height', 'WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight', 'Rings']
Shape: (4177, 11)


,Sex_F,Sex_I,Sex_M,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,0,0,1,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,0,0,1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,1,0,0,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,0,0,1,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,0,1,0,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


### 4. Outlier Clipping

In [34]:
def clip_outliers_iqr(df: pd.DataFrame, columns: list, factor: float = 1.5) -> pd.DataFrame:
    """Clip outliers using IQR method for specified columns."""
    df_clipped = df.copy()
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - factor * IQR
        upper_bound = Q3 + factor * IQR
        df_clipped[col] = np.clip(df[col], lower_bound, upper_bound)
    return df_clipped

# Áp dụng clipping cho các cột numeric (size và weight)
clipped_df = clip_outliers_iqr(encoded, numeric_cols)
print('Shape sau clipping:', clipped_df.shape)
print('Mô tả thống kê sau clipping:')
display(clipped_df[numeric_cols].describe())

Shape sau clipping: (4177, 11)
Mô tả thống kê sau clipping:


,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
count,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000
mean,0.524373,0.408271,0.139318,0.827425,0.357809,0.180285,0.238049
std,0.118984,0.098159,0.038272,0.486184,0.216794,0.108577,0.136487
min,0.202500,0.155000,0.040000,0.002000,0.001000,0.000500,0.001500
25%,0.450000,0.350000,0.115000,0.441500,0.186000,0.093500,0.130000
50%,0.545000,0.425000,0.140000,0.799500,0.336000,0.171000,0.234000
75%,0.615000,0.480000,0.165000,1.153000,0.502000,0.253000,0.329000
max,0.815000,0.650000,0.240000,2.220250,0.976000,0.492250,0.627500


### 5. Robust Log Transform

In [35]:
# Áp dụng log1p transform cho các cột weight và size (robust transform)
log_transformed_df = apply_log1p_to_columns(clipped_df, size_cols + weight_cols)
print('Shape sau log transform:', log_transformed_df.shape)
print('Mô tả thống kê sau log transform:')
display(log_transformed_df[numeric_cols].describe())

Shape sau log transform: (4177, 11)
Mô tả thống kê sau log transform:


,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
count,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000,4177.000000
mean,0.418416,0.339855,0.129862,0.567512,0.293471,0.161615,0.207573
std,0.080436,0.071436,0.033758,0.267384,0.156713,0.090584,0.108816
min,0.184403,0.144100,0.039221,0.001998,0.001000,0.000500,0.001499
25%,0.371564,0.300105,0.108854,0.365684,0.170586,0.089384,0.122218
50%,0.435024,0.354172,0.131028,0.587509,0.289680,0.157858,0.210261
75%,0.479335,0.392042,0.152721,0.766862,0.406798,0.225541,0.284427
max,0.596085,0.500775,0.215111,1.169459,0.681075,0.400285,0.487045


### Lưu dữ liệu preprocssed

In [36]:
# Lưu hai tập dữ liệu processed
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

clipped_path = processed_dir / 'abalone_clipped.csv'
log_transformed_path = processed_dir / 'abalone_log_transformed.csv'

clipped_df.to_csv(clipped_path, index=False)
log_transformed_df.to_csv(log_transformed_path, index=False)

print('Đã lưu dữ liệu clipped')
print('Đã lưu dữ liệu log transformed')

Đã lưu dữ liệu clipped
Đã lưu dữ liệu log transformed
